# Project 8 — Module 9: Fundamentos de Big Data
## Lesson 01 — Business Understanding

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 01 — Business Understanding |
| **Module** | M9 — Fundamentos de Big Data (Alkemy Bootcamp) |
| **Dataset** | Brazilian E-Commerce (Olist) + Synthetic clickstream |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook establishes the business context for RetailMax's Big Data analytics
> pipeline. It defines the business problem (fragmented analytics unable to scale),
> maps the 5V of Big Data to RetailMax's data sources, proposes a pipeline
> architecture (PySpark local → AWS EMR Serverless), and produces a conceptual
> report that guides all subsequent lessons. The key output is the Problem Statement
> Canvas and architecture diagram that frame the entire project.

## Table of Contents

1. [CRISP-DM Phase 01 — Business Understanding](#1-crisp-dm-phase-01--business-understanding)
2. [Problem Statement Canvas](#2-problem-statement-canvas)
3. [The 5V of Big Data — RetailMax Context](#3-the-5v-of-big-data--retailmax-context)
4. [Data Sources Identification](#4-data-sources-identification)
5. [Pipeline Architecture Proposal](#5-pipeline-architecture-proposal)
6. [Lesson-to-Notebook Mapping (CRISP-DM)](#6-lesson-to-notebook-mapping-crisp-dm)
7. [Personal Perspective — ICI Background](#7-personal-perspective--ici-background)
8. [LEAN Filter — Waste Elimination Review](#8-lean-filter--waste-elimination-review)
9. [Decisions Log — Lesson 01](#9-decisions-log--lesson-01)
10. [Next Steps — Lesson 02 Preview](#10-next-steps--lesson-02-preview)

---

## 1. CRISP-DM Phase 01 — Business Understanding

### Business Context

**RetailMax** is a large-scale e-commerce platform processing millions of daily
transactions across multiple product categories. The company collects data from
three primary sources: purchase transactions, product reviews and ratings, and
user browsing behavior (clickstream).

### Current Pain Point

The analytics department faces two critical bottlenecks:

1. **Scalability:** Traditional tools (pandas, Excel) cannot process the full
   transaction history — analyses run on samples, losing signal in the tail.
2. **Fragmentation:** Data lives in separate systems (transactional DB, review
   platform, web analytics) — no unified view of customer behavior exists.

### Business Objective

Build a **scalable Big Data + ML pipeline** using Apache Spark that:

- Ingests and unifies structured data (transactions, payments) with
  semi/unstructured data (reviews with free text, clickstream logs).
- Generates business metrics at scale (revenue by category, top products,
  customer lifetime patterns).
- Classifies customers as High Value vs. Low Value (supervised — Logistic
  Regression) and segments them for targeted marketing (unsupervised — K-Means).
- Delivers actionable insights to the marketing team.

---

## 2. Problem Statement Canvas

| Element | Description |
|---|---|
| **Business Problem** | RetailMax's analytics process is slow and fragmented — data stored across multiple systems, traditional ML tools cannot scale to the full dataset volume. Marketing decisions are based on sampled data, missing patterns in long-tail customer segments. |
| **Business Impact** | Suboptimal marketing spend — campaigns target broad segments instead of high-value micro-segments. Estimated 15–25% waste in retention budget due to untargeted offers. Review sentiment is not systematically incorporated into product strategy. |
| **Decision to Support** | Marketing needs to decide: (1) which customers receive high-value retention offers vs. standard promotions, and (2) which product categories to prioritize based on full-scale transaction + review analysis. |
| **Analytical Question** | Can we classify customers as High Value vs. Low Value using transaction frequency, ticket size, review sentiment, and browsing behavior — at scale — and identify natural customer segments for targeted campaigns? |
| **Success Metrics** | Logistic Regression: AUC-ROC > 0.75 on test set. K-Means: Silhouette Score > 0.40 with interpretable clusters. Pipeline processes 2M+ records without memory errors. Parquet output reduces storage by > 50% vs. CSV. |
| **Proposed Approach** | PySpark pipeline: ingest Olist data + synthetic clickstream → RDD transformations → Spark SQL metrics → MLlib (LogReg for classification, K-Means for segmentation). Local development + optional AWS EMR Serverless validation. |

---

## 3. The 5V of Big Data — RetailMax Context

| V | Definition | RetailMax Application | Scale |
|---|---|---|---|
| **Volume** | Amount of data generated and stored | 100K+ orders (Olist base) scaled to 1M+ transactions, 500K clickstream events, 100K reviews, 1M geolocation records | ~2M+ total records across all sources |
| **Velocity** | Speed at which data is generated | In production: real-time transactions and clickstream events. In this project: batch processing of historical data simulating near-real-time pipeline readiness | Batch (historical) with architecture designed for streaming extension |
| **Variety** | Different types and formats of data | Structured (transactions, payments, products — CSV/Parquet), semi-structured (reviews with text + rating), synthetic (clickstream logs) | 3 distinct source types, 9+ tables |
| **Veracity** | Data quality and trustworthiness | Olist is real anonymized commercial data (2016–2018). Synthetic clickstream is documented as simulated. Review text has been anonymized (company names replaced with fictional references) | Real data with known limitations documented |
| **Value** | Business insights derived from data | Customer classification (High/Low Value), segmentation for marketing, revenue analysis by category, review-informed product strategy | Direct marketing ROI improvement |

---

## 4. Data Sources Identification

### Source 1: Olist Brazilian E-Commerce Dataset (Real — Kaggle)

| Field | Details |
|---|---|
| **Dataset** | [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) |
| **Provider** | Kaggle |
| **License** | CC BY-NC-SA 4.0 |
| **Records** | ~100K orders, 9 interrelated CSV tables |
| **Period** | 2016–2018 |
| **Accessed** | April 2026 |

**Tables and their role in the pipeline:**

| Table | Records | Type | Pipeline Role |
|---|---|---|---|
| `olist_orders_dataset` | 100K | Structured | Core — order status, timestamps |
| `olist_order_items_dataset` | 113K | Structured | Product-level detail per order |
| `olist_order_payments_dataset` | 104K | Structured | Payment methods, installments |
| `olist_order_reviews_dataset` | 100K | Semi/Unstructured | Rating (1–5) + review text |
| `olist_products_dataset` | 33K | Structured | Product catalog, categories |
| `olist_customers_dataset` | 100K | Structured | Customer ID + location |
| `olist_sellers_dataset` | 3K | Structured | Seller information |
| `olist_geolocation_dataset` | 1M | Structured | Zip code → lat/lng coordinates |
| `product_category_name_translation` | 71 | Lookup | Portuguese → English categories |

### Source 2: Synthetic Clickstream Data (Generated with PySpark)

| Field | Details |
|---|---|
| **Dataset** | Synthetic browsing behavior |
| **Generator** | `spark.range()` + distributed column functions |
| **Records** | ~500K events |
| **Rationale** | Completes the 3rd data source required by the consigna (navigation behavior). Documents the ability to generate data at scale using Spark's distributed engine — never Python loops + `createDataFrame` |

**Schema:**

| Column | Type | Description |
|---|---|---|
| `session_id` | STRING | Unique browsing session identifier |
| `customer_id` | STRING | Links to Olist customers |
| `page_viewed` | STRING | Page type (home, product, category, cart, checkout) |
| `time_spent_seconds` | INT | Duration on page (5–300s) |
| `device` | STRING | Device type (desktop, mobile, tablet) |
| `event_timestamp` | TIMESTAMP | Browsing event timestamp |

> **LEAN note:** Synthetic data serves as a low-cost simulation environment.
> This is explicitly documented — no ambiguity about data provenance.

---

## 5. Pipeline Architecture Proposal

### Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                    RetailMax Analytics Pipeline                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  DATA SOURCES              PROCESSING              OUTPUT           │
│  ┌──────────┐            ┌──────────────┐       ┌──────────┐       │
│  │Olist CSVs│──┐         │              │       │ Parquet  │       │
│  │(9 tables)│  │    ┌───▶│  Spark SQL   │──────▶│  Files   │       │
│  └──────────┘  │    │    │  DataFrames  │       └──────────┘       │
│                ├───▶│    │              │                           │
│  ┌──────────┐  │    │    └──────────────┘       ┌──────────┐       │
│  │Synthetic │  │    │                           │  MLlib   │       │
│  │Clickstrm │──┘    │    ┌──────────────┐  ┌───▶│ LogReg + │       │
│  └──────────┘       │    │              │  │    │ K-Means  │       │
│                     └───▶│     RDD      │──┘    └──────────┘       │
│                          │Transformtions│                           │
│                          │              │       ┌──────────┐       │
│                          └──────────────┘       │ Reports  │       │
│                                                 │ Insights │       │
│                                                 └──────────┘       │
│                                                                     │
│  ENVIRONMENT: PySpark 4.1.1 (local) → AWS EMR Serverless (optional)│
│  STORAGE: CSV (raw) → Parquet (processed/final)                    │
│  JAVA: Temurin 17                                                  │
└─────────────────────────────────────────────────────────────────────┘
```

### Technology Stack

| Component | Technology | Rationale |
|---|---|---|
| Processing Engine | PySpark 4.1.1 | Distributed processing, consigna requirement |
| ML Library | Spark MLlib | Scalable ML — LogReg + K-Means at Big Data scale |
| Storage (raw) | CSV | Original Olist format, human-readable |
| Storage (processed) | Apache Parquet | Columnar, compressed, ~50–70% smaller than CSV |
| JVM Runtime | Java 17 (Temurin) | PySpark 4.x requirement |
| Cloud (optional) | AWS EMR Serverless + S3 | Validates scalability beyond local machine |
| Language | Python 3.12 | Shared `.venv` at portfolio root |

### Design Decisions

| Decision | Rationale | Alternative Considered |
|---|---|---|
| Hybrid local + cloud | Develop locally for speed, validate on EMR for scale proof | Full cloud (expensive), full local (no scale proof) |
| Parquet over CSV for intermediate storage | 50–70% compression, predicate pushdown, schema enforcement | JSON (verbose), Delta Lake (overkill for bootcamp) |
| Olist real data + synthetic clickstream | Real data adds portfolio credibility; synthetic covers missing source | 100% synthetic (less credible), 100% real (no clickstream available) |
| `spark.range()` for data generation | Distributed generation — avoids driver-side bottleneck | Python loops + `createDataFrame` (antipattern — driver OOM risk) |

---

## 6. Lesson-to-Notebook Mapping (CRISP-DM)

| Notebook | CRISP-DM Phase | Consigna Lesson | Key Deliverable |
|---|---|---|---|
| 01 — Business Understanding | Phase 1 | L1: Fundamentos de Big Data | Problem Statement Canvas, 5V analysis, architecture diagram, conceptual report |
| 02 — Data Understanding | Phase 2 | L2: Apache Spark Intro + Config | SparkSession config, data loading in RDDs, basic actions (count, take), initial exploration |
| 03 — Data Preparation | Phase 3 | L3: RDD Transformations + Actions | RDDs, Pair RDDs, map/filter/flatMap/sortBy, linaje documentation, DAG explanation |
| 04 — Modeling | Phase 4 | L4: Spark SQL + DataFrames, L5: MLlib | DataFrames with explicit schemas, SQL business metrics, Parquet output, MLlib pipeline (LogReg + K-Means) |
| 05 — Evaluation | Phase 5 | L5: MLlib (continuation) | Model metrics evaluation, cluster interpretation, marketing insights report |
| 06 — Deployment | Phase 6 | Integration | Final report (PDF), pipeline documentation, consigna compliance verification, retrospective |

> **CRISP-DM scope decision:** This project implements all 6 phases. L4 (Spark SQL)
> is grouped with L5 (MLlib) in the Modeling notebook because Spark SQL
> transformations are the feature engineering step that feeds directly into the
> ML pipeline — separating them would break the analytical flow.

---

## 7. Personal Perspective — ICI Background

This project connects directly to industrial engineering principles:

- **Pipeline as production line:** Each Spark transformation is analogous to a
  workstation in a manufacturing process. The DAG (Directed Acyclic Graph) is
  the plant layout — Spark's optimizer eliminates redundant steps just as
  Lean eliminates waste in a production flow.

- **Lazy evaluation = pull system:** Spark only executes transformations when
  an action triggers them — this mirrors JIT (Just-In-Time) manufacturing
  where production only starts when demand (the action) requires it.

- **Cache/persist = buffer inventory:** Strategic caching of intermediate
  results is analogous to placing buffer stock at bottleneck stations —
  it costs memory (inventory) but prevents recomputation (rework).

- **Partitioning = work distribution:** Spark distributes data across
  partitions like assigning work orders to parallel production cells —
  balancing load is key to throughput.

These analogies are not decorative — they reflect how an operations
professional thinks about data pipelines and why this perspective
adds value in analytics roles.

---

## 8. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Does every analysis link to a business decision? | ✅ All sections connect to the pipeline design and marketing classification goal | Proceed |
| Is there redundancy between sections? | ✅ No overlap — each section adds a unique dimension (business context, data, architecture) | Proceed |
| Is there a simpler way to answer the question? | ✅ This is the minimum viable business understanding — no unnecessary theory | Proceed |
| Is the scope appropriate for Phase 1? | ✅ Conceptual only — no code, no data processing, no modeling. That belongs in NB 02–05 | Proceed |

---

## 9. Decisions Log — Lesson 01

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| D1 | Use Olist dataset as primary data source | Real commercial data adds portfolio credibility; 9 tables provide variety; CC BY-NC-SA 4.0 license is compatible | Online Retail UK (only 540K, no reviews), 100% synthetic (less credible) | ✅ |
| D2 | Generate clickstream data with `spark.range()` | Covers consigna requirement for navigation data; demonstrates distributed generation skill | Python loops + `createDataFrame` (antipattern), skip navigation data (incomplete consigna coverage) | ✅ |
| D3 | Hybrid local + AWS EMR architecture | Cost-effective development (~$2–5 USD for EMR validation) while proving cloud scalability | Full local only (no scale proof), full EMR (expensive for development) | ✅ |
| D4 | Map 5 consigna lessons to 6 CRISP-DM notebooks | Maintains standard CRISP-DM structure; L4+L5 naturally merge into Modeling+Evaluation phases | 5 notebooks (breaks CRISP-DM standard), 1 notebook per lesson (loses framework coherence) | ✅ |
| D5 | Parquet as intermediate storage format | Columnar compression (50–70% vs CSV), schema enforcement, predicate pushdown for Spark SQL | CSV (no compression), Delta Lake (overkill for bootcamp scope) | ✅ |

---

## 10. Next Steps — Lesson 02 Preview

| Priority | Next Step | Target |
|---|---|---|
| 🔴 Immediate | Download Olist dataset from Kaggle and place CSVs in `data/raw/` | Before starting NB 02 |
| 🔴 Immediate | Configure SparkSession with PySpark 4.1.1 + Java 17 Temurin | NB 02 — Data Understanding |
| 🟡 Short-term | Load all 9 Olist tables as RDDs, validate with `count()` and `take()` | NB 02 |
| 🟡 Short-term | Generate synthetic clickstream data using `spark.range()` | NB 02 or NB 03 |
| 🟢 Long-term | Set up AWS account for EMR Serverless validation | Before NB 06 |

---

**Next Phase →** [02 — Data Understanding](./02_data_understanding.ipynb)